# Echo Cancellation

Build an echo, cancel it, break the canceller with double talk, and fix it. Predict each number before you run the cell.

In [ ]:
import sys
sys.path.insert(0, '../code')
import numpy as np
import main

s = main.scenario(seconds=3.0)
residual, w = main.nlms_echo_canceller(s['far'], s['mic'])
erle = main.erle_db(s['echo'], residual - s['noise'], frame=800)
np.round(erle, 1), round(main.misalignment_db(s['path'], w), 1)

## Regularization

Try `eps` values from `1e-4` to `1.0`. Why does a tiny `eps` hurt in noise but not without noise?

In [ ]:
for eps in (1e-4, 1e-2, 1e-1, 1.0):
    residual, w = main.nlms_echo_canceller(s['far'], s['mic'], eps=eps)
    late = np.nanmedian(main.erle_db(s['echo'], residual - s['noise'], frame=800)[-8:])
    print(f'eps={eps:<6} ERLE in the last 0.8 s: {late:5.1f} dB')

## Double talk

The caller speaks from 2.0 s to 3.0 s. Compare the learned filter with and without the Geigel freeze.

In [ ]:
dt = main.scenario(seconds=4.0, double_talk=(2.0, 3.0))
flag = main.geigel_double_talk(dt['far'], dt['mic'])
_, w_free = main.nlms_echo_canceller(dt['far'], dt['mic'])
_, w_held = main.nlms_echo_canceller(dt['far'], dt['mic'], freeze=flag)
print('misalignment, always adapting:', round(main.misalignment_db(dt['path'], w_free), 1), 'dB')
print('misalignment, frozen on double talk:', round(main.misalignment_db(dt['path'], w_held), 1), 'dB')

## False barge-in

How loud does the echo have to be before an energy VAD fires? Change `erl_db` (6 dB is a poor hybrid, 0 dB a speakerphone).

In [ ]:
for erl in (20, 10, 6, 0):
    e = main.scenario(seconds=3.0, erl_db=erl)
    residual, _ = main.nlms_echo_canceller(e['far'], e['mic'])
    print(f'ERL {erl:>2} dB: {main.false_barge_in_frames(e["mic"]):3d} false frames raw, '
          f'{main.false_barge_in_frames(residual[8000:]):3d} after 1 s of AEC')